## **Notebook de Exploração dos Dados**

**O objetivo desse notebook é fazer uma análise exploratória dos arquivos que serão utilizados neste projeto.**

Como entradas de dados, estão sendo utilizadas:

Viagens de Táxi da Cidade de Nova York: 
-     yellow_tripdata_2025-01.parquet
-     yellow_tripdata_2025-02.parquet
-     yellow_tripdata_2025-03.parquet

Zonas da Cidade de Nova York: 
-     taxi_zone_lookup.csv

**Importando as bibliotecas**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Criando os DataFrames a partir dos arquivos:**

- Viagens (arquivos Parquet):
    - janeiro
    - fevereiro
    - março
- Zonas (arquivo CSV)

In [0]:
path = [
    "/Volumes/nyc_taxi_data/raw/landing/trips/yellow_tripdata_2025-01.parquet",
    "/Volumes/nyc_taxi_data/raw/landing/trips/yellow_tripdata_2025-02.parquet",
    "/Volumes/nyc_taxi_data/raw/landing/trips/yellow_tripdata_2025-03.parquet"
]


#lendo o dataframe a partir do path
df_viagens = spark.read.parquet(*path)

df_zonas = spark.read.csv("/Volumes/nyc_taxi_data/raw/landing/lookup/taxi_zone_lookup.csv", inferSchema=True, header=True)



**Conhecendo os dados:**

In [0]:
df_viagens.show(50)

df_zonas.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-03-01 00:17:16|  2025-03-01 00:25:52|              1|          0.9|         1|                 N|         140|    

**Verificando o schema do DataFrame Viagens:**

In [0]:
df_viagens.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



**Verificando o schema do DataFrame Zonas:**

In [0]:
df_zonas.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



**Checando valores nulos do DataFrame Viagens:**

In [0]:
for coluna in df_viagens.columns:
    print(coluna, df_viagens.filter(df_viagens[coluna].isNull()).count())

VendorID 0
tpep_pickup_datetime 0
tpep_dropoff_datetime 0
passenger_count 2263749
trip_distance 0
RatecodeID 2263749
store_and_fwd_flag 2263749
PULocationID 0
DOLocationID 0
payment_type 0
fare_amount 0
extra 0
mta_tax 0
tip_amount 0
tolls_amount 0
improvement_surcharge 0
total_amount 0
congestion_surcharge 2263749
Airport_fee 2263749
cbd_congestion_fee 0


**Checando valores nulos do DataFrame Zonas:**

In [0]:
for coluna in df_zonas.columns:
    print(coluna, df_zonas.filter(df_zonas[coluna].isNull()).count())

LocationID 0
Borough 0
Zone 0
service_zone 0


**Checando se há valores duplicados nos Dataframes:**

In [0]:
total_valores_df_viagens = df_viagens.count()
total_valores_distintos_df_viagens = df_viagens.distinct().count()

total_valores_df_zonas = df_zonas.count()
total_valores_distintos_df_zonas = df_zonas.distinct().count()



print(f'O total de valores no DF Viagens (incluindo janeiro, fevereiro e março) é {total_valores_df_viagens}')
print(f'O total de valores distintos no DF Viagens (incluindo janeiro, fevereiro e março) é {total_valores_distintos_df_viagens}')
print(f'O total de valores no DF Zonas é {total_valores_df_zonas}')
print(f'O total de valores distintos no DF Zonas é {total_valores_distintos_df_zonas}') 

O total de valores no DF Viagens (incluindo janeiro, fevereiro e março) é 11198026
O total de valores distintos no DF Viagens (incluindo janeiro, fevereiro e março) é 11198026
O total de valores no DF Zonas é 265
O total de valores distintos no DF Zonas é 265


### Demonstração de RDDs - Limitação do ambiente

Foi avaliada a utilização de RDDs (Resilient Distributed Datasets) e sua conversão para DataFrames como parte da demonstração das APIs fundamentais do Apache Spark.

Entretanto, o ambiente utilizado neste projeto é baseado em Databricks Serverless, que não disponibiliza acesso direto ao `SparkContext`. Ao tentar acessar `spark.sparkContext`, o ambiente retornou o erro `JVM_ATTRIBUTE_NOT_SUPPORTED`.

Dessa forma, não foi possível implementar a criação explícita de RDDs neste ambiente. O pipeline foi desenvolvido utilizando a API de DataFrames e Spark SQL, que representam a abordagem principal adotada no projeto para processamento distribuído e integração com os mecanismos de otimização do Spark.